# fine-tuning using your in-house assay

This tutorial is for users who have already run a functional assay and would like to answer the question "what if I had run my assay with other drugs, other CRISPR treatments, or in other cell lines?"

For example, [Tieu et al (2024)](https://doi.org/10.1016/j.cell.2024.01.035) run a combinatorial CAR-T transduction with 24 guides for a total of 576 pairwise combinations. We can fine-tune Prophet on this dataset and make predictions for genes spanning the entire genome and additional combinations therein.

In [1]:
import pandas as pd
import yaml
from prophet import Prophet, set_config

/home/icb/yuge.ji/miniconda3/envs/prophetv2_copy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We load in a config file to automatically get the file paths for the embeddings which were used, but any of the files in `embeddings` can be passed.

In [ ]:
with open('config_file_finetuning.yaml', 'r') as f:
    config = set_config(yaml.safe_load(f))

We'll randomly select one of the pretrained model checkpoints to train on here. For more robust results, we recommend training with several checkpoints and taking the ensemble prediction.

In [ ]:
# replace this with the path to the model checkpoint you want to use
path = './ckpts/epoch=15-step=12352.ckpt'

In [7]:
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=config.phenotype_prior,
    model_pth=path,
)

/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])


returning trained model!
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=128, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=128, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=128, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=128, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=128, out_features=128, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=128, out_features=128, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=128, out_features=1, bias=True)
)


/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])


Suppose we have some small molecules, some cell lines we would like to test them in, and we're interested in measuring their relative IC50. We can pass in lists of these inputs, and Prophet will return predictions for all combinations:

#### Making predictions by passing all treatments, cell lines, and phenotypes you want to run

This format can be useful when running large combinatorial screens in silico, as it splits the experiments up into batches to help prevent memory errors.

In [8]:
iv_list = [
    'oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc4=c(c=c(c=c4)i)f)=o',
    'cc(nc1=cc=cc(n(c2=o)c(c(c(n2c3cc3)=o)=c(n4c)nc5=cc=c(c=c5f)i)=c(c4=o)c)=c1)=o',
    'fc1=cc=c(c(f)=c1c(c2=cnc3=nc=c(c=c32)c4=cc=c(c=c4)cl)=o)ns(ccc)(=o)=o',
    'cs(=o)c'  # DMSO
]
cl_list = ['A375','UACC62','WM983B','MALME3M','A2058','WM793','HT144','RPMI7951','WM1799','LOXIMVI','WM2664','WM88','G361','SKMEL24','WM115', 'SKMEL2', 'SKMEL1', 'HMCB', 'MDAMB435S', 'UACC257']
ph_list = ['GDSC']

In [9]:
# predict with lists of treatments and cell lines
df = model.train(
    target_ivs=iv_list,
    target_cls=cl_list,
    target_phs=ph_list,
    iv_col=['iv1', 'iv2'],  # pass to turn on combinatorial predictions
    num_iterations=1, save=False,
    model_config=config,
)
df

There are 1 iterations


  0%|                                                                                                                                                           | 0/1 [00:00<?, ?it/s]

Removing 0 such as [] from ['iv1', 'iv2']. 200 rows remaining.
Removing 0 such as [] from ['cell_line']. 200 rows remaining.


/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])
/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])
/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.pro

Predicting DataLoader 0:   0%|                                                                                                                                  | 0/1 [00:00<?, ?it/s]

/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/torch/nn/modules/transformer.py:408: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:14<00:00,  0.07it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:17<00:00, 77.30s/it]


,iv1,iv2,cell_line,phenotype,iv1+iv2,value,pred
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.221658
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.234035
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.263363
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.290998
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,GDSC,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,_,0.341044
...,...,...,...,...,...,...,...
252,cs(=o)c,cs(=o)c,SKMEL2,GDSC,cs(=o)c+cs(=o)c,_,0.627905
253,cs(=o)c,cs(=o)c,SKMEL1,GDSCcomb,cs(=o)c+cs(=o)c,_,0.639536
254,cs(=o)c,cs(=o)c,HMCB,PRISM,cs(=o)c+cs(=o)c,_,0.627371
255,cs(=o)c,cs(=o)c,MDAMB435S,inhouse,cs(=o)c+cs(=o)c,_,0.615494


#### Making predictions for a specific set of treatments, cell lines, and phenotypes

If we're interested in only a subset of the experimental matrix, we can also pass in a custom dataframe. (This is the recommended usage, as users understand exactly the list being predicted.)

In [14]:
# construct a dataframe containing the experiments we want to run
experiments_df = pd.MultiIndex.from_product([
    iv_list,
    cl_list,
], names=['iv1', 'cell_line'])
experiments_df = experiments_df.to_frame(index=False).reset_index(drop=True)
experiments_df

/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/indexes/multi.py:643: DeprecationWarning: `cumproduct` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `cumprod` instead.
  codes = cartesian_product(codes)
/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/indexes/multi.py:643: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  codes = cartesian_product(codes)


,iv1,cell_line
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058
...,...,...
75,cs(=o)c,SKMEL2
76,cs(=o)c,SKMEL1
77,cs(=o)c,HMCB
78,cs(=o)c,MDAMB435S


In [13]:
input_df = experiments_df.copy()
input_df['iv2'] = 'cs(=o)c'  # DMSO
input_df['phenotype'] = 'GDSC'
df = model.predict(input_df, num_iterations=1, save=False)
df

/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])


There are 1 iterations


  0%|                                                                                                                                                           | 0/1 [00:00<?, ?it/s]/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/algorithms.py:522: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  common = np.find_common_type([values.dtype, comps_array.dtype], [])
/home/icb/yuge.ji/miniconda3/envs/prophet_api/lib/python3.12/site-packages/pandas/core/dtypes/cast.py:1641: DeprecationWarning: np.find_common_type is deprecated.  Please use `np.result_type` or `np.promote_types`.
See https://numpy.org/devdocs/release/1.25.0-notes.html and the docs for more information.  (Deprecated NumPy 1.25)
  return np.find_common_type(types, [])
/home/icb/yuge.ji/miniconda3/envs/prophet_api/li

Predicting DataLoader 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 32.75it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:14<00:00, 14.35s/it]


,iv1,cell_line,iv2,phenotype,pred
0,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A375,cs(=o)c,GDSC,0.479495
1,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,UACC62,cs(=o)c,GDSC,0.434696
2,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,WM983B,cs(=o)c,GDSC,0.501254
3,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,MALME3M,cs(=o)c,GDSC,0.529391
4,oc1([c@h]2ncccc2)cn(c1)c(c3=c(c(f)=c(c=c3)f)nc...,A2058,cs(=o)c,GDSC,0.528486
...,...,...,...,...,...
132,cs(=o)c,SKMEL2,cs(=o)c,GDSC,0.627905
133,cs(=o)c,SKMEL1,cs(=o)c,GDSCcomb,0.639536
134,cs(=o)c,HMCB,cs(=o)c,PRISM,0.627371
135,cs(=o)c,MDAMB435S,cs(=o)c,inhouse,0.615494
